In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/processed/train.csv")
test  = pd.read_csv("../data/processed/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (77108, 42)
Test shape: (17014, 41)


In [2]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

In [3]:
print("=== Missing Values in Date Columns ===")
print(train[date_cols].isna().sum())

=== Missing Values in Date Columns ===
order_purchase_timestamp          0
order_approved_at                12
order_delivered_carrier_date      1
order_delivered_customer_date     0
order_estimated_delivery_date     0
dtype: int64


In [4]:
for col in date_cols:
    train[col] = pd.to_datetime(train[col])
    test[col]  = pd.to_datetime(test[col])

print("✅ Date columns converted")
print(train[date_cols].dtypes)

✅ Date columns converted
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [5]:
train['is_late'] = (
    train['order_delivered_customer_date'] > train['order_estimated_delivery_date']
).astype(int)

# Mark missing delivery dates as NaN
missing_delivery = train['order_delivered_customer_date'].isna()
train.loc[missing_delivery, 'is_late'] = np.nan

print("=== is_late created ===")
print(train['is_late'].value_counts(dropna=False))

=== is_late created ===
is_late
0.0    70958
1.0     6150
Name: count, dtype: int64


In [6]:
# See what order_status those missing rows have
print(train[missing_delivery]['order_status'].value_counts())
before = train.shape[0]
train = train.dropna(subset=['order_delivered_customer_date'])
after = train.shape[0]

print(f"\n✅ Undelivered orders dropped")
print(f"Removed: {before - after} rows")
print(f"Remaining rows: {after}")
print(f"Train shape after drop: {train.shape}")

Series([], Name: count, dtype: int64)

✅ Undelivered orders dropped
Removed: 0 rows
Remaining rows: 77108
Train shape after drop: (77108, 42)


In [7]:
counts = train['is_late'].value_counts()
pct = train['is_late'].value_counts(normalize=True) * 100
print(f"Late    (1): {counts[1]} rows  ({pct[1]:.1f}%)")
print(f"On time (0): {counts[0]} rows  ({pct[0]:.1f}%)")

Late    (1): 6150 rows  (8.0%)
On time (0): 70958 rows  (92.0%)


In [8]:
leaky_columns = [
    'order_delivered_customer_date',
    'order_delivered_carrier_date',
    'order_status'
]

print("=== LEAKY COLUMNS — NEVER USE AS FEATURES ===")
for col in leaky_columns:
    print(f"  don't use {col}")

=== LEAKY COLUMNS — NEVER USE AS FEATURES ===
  don't use order_delivered_customer_date
  don't use order_delivered_carrier_date
  don't use order_status


# Basic Feature Engineering

In [9]:
# Create purchase month feature

train['purchase_month'] = train['order_purchase_timestamp'].dt.month
test['purchase_month'] = test['order_purchase_timestamp'].dt.month

In [10]:
train[['order_purchase_timestamp', 'purchase_month']].head()

,order_purchase_timestamp,purchase_month
0,2018-04-30 19:56:03,4
1,2018-03-11 19:30:33,3
2,2018-03-02 15:27:23,3
3,2018-08-23 19:28:09,8
4,2018-04-16 13:14:09,4


In [11]:
#orders placed in each month
train['purchase_month'].value_counts().sort_index()

purchase_month
1     6271
2     6548
3     7619
4     7266
5     8376
6     7366
7     7894
8     8353
9     3370
10    3882
11    5892
12    4271
Name: count, dtype: int64

In [12]:
# Create purchase day feature

train['purchase_day'] = train['order_purchase_timestamp'].dt.day_name()
test['purchase_day'] = test['order_purchase_timestamp'].dt.day_name()

In [13]:
train[['order_purchase_timestamp', 'purchase_day']].head()

,order_purchase_timestamp,purchase_day
0,2018-04-30 19:56:03,Monday
1,2018-03-11 19:30:33,Sunday
2,2018-03-02 15:27:23,Friday
3,2018-08-23 19:28:09,Thursday
4,2018-04-16 13:14:09,Monday


In [14]:
#orders placed in each day
train['purchase_day'].value_counts()

purchase_day
Tuesday      12640
Monday       12638
Wednesday    12090
Thursday     11549
Friday       10860
Sunday        9014
Saturday      8317
Name: count, dtype: int64

In [15]:
# Create weekend feature

train['is_weekend'] = train['purchase_day'].isin(['Saturday', 'Sunday']).astype(int)

test['is_weekend'] = test['purchase_day'].isin(['Saturday', 'Sunday']).astype(int)

In [16]:
train[['purchase_day', 'is_weekend']].head(10)

,purchase_day,is_weekend
0,Monday,0
1,Sunday,1
2,Friday,0
3,Thursday,0
4,Monday,0
5,Sunday,1
6,Monday,0
7,Thursday,0
8,Thursday,0
9,Wednesday,0


In [17]:
#orders placed in weekend and no
train['is_weekend'].value_counts()

is_weekend
0    59777
1    17331
Name: count, dtype: int64

In [18]:
# Create estimated delivery days feature

train['estimated_delivery_days'] = (
    train['order_estimated_delivery_date'] -
    train['order_purchase_timestamp']
).dt.days

test['estimated_delivery_days'] = (
    test['order_estimated_delivery_date'] -
    test['order_purchase_timestamp']
).dt.days

In [19]:
train[
    [
        'order_purchase_timestamp',
        'order_estimated_delivery_date',
        'estimated_delivery_days'
    ]
].head()

,order_purchase_timestamp,order_estimated_delivery_date,estimated_delivery_days
0,2018-04-30 19:56:03,2018-05-29,28
1,2018-03-11 19:30:33,2018-04-03,22
2,2018-03-02 15:27:23,2018-03-28,25
3,2018-08-23 19:28:09,2018-09-18,25
4,2018-04-16 13:14:09,2018-05-02,15


In [20]:
train['estimated_delivery_days'].describe()

count    77108.000000
mean        23.420929
std          8.837262
min          2.000000
25%         18.000000
50%         23.000000
75%         28.000000
max        155.000000
Name: estimated_delivery_days, dtype: float64

In [21]:
train['estimated_delivery_days'].value_counts().sort_index().head(20)

estimated_delivery_days
2      144
3      112
4      255
5      233
6      261
7      689
8      539
9      917
10     995
11    1359
12    2056
13    2012
14    1620
15    2053
16    2246
17    2514
18    2738
19    3736
20    3779
21    4612
Name: count, dtype: int64

In [22]:
import numpy as np

# Create freight-to-price ratio feature

train['freight_to_price_ratio'] = np.where(
    train['price'] != 0,
    train['freight_value'] / train['price'],
    0
)

test['freight_to_price_ratio'] = np.where(
    test['price'] != 0,
    test['freight_value'] / test['price'],
    0
)

In [23]:
train[['price','freight_value','freight_to_price_ratio']].head()

,price,freight_value,freight_to_price_ratio
0,119.00,19.74,0.165882
1,44.90,22.93,0.510690
2,65.90,16.90,0.256449
3,14.49,18.23,1.258109
4,109.99,23.25,0.211383


In [24]:
train['freight_to_price_ratio'].describe()

count    77108.000000
mean         0.320523
std          0.337725
min          0.000000
25%          0.134832
50%          0.231701
75%          0.394874
max         26.235294
Name: freight_to_price_ratio, dtype: float64

In [25]:
# Create same_state feature

train['same_state'] = (
    train['seller_state'] == train['customer_state']
).astype(int)

test['same_state'] = (
    test['seller_state'] == test['customer_state']
).astype(int)

In [26]:
train[['seller_state','customer_state','same_state']].head(10)

,seller_state,customer_state,same_state
0,MG,ES,0
1,SP,MG,0
2,SC,RJ,0
3,SP,RJ,0
4,SP,SP,1
5,SP,ES,0
6,MG,SP,0
7,SP,SC,0
8,SP,SP,1
9,SC,RJ,0


In [27]:
train['same_state'].value_counts()

same_state
0    49176
1    27932
Name: count, dtype: int64

In [28]:
train.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,is_weekend,estimated_delivery_days,freight_to_price_ratio,same_state,seller_delay_rate,seller_lat,seller_lng,customer_lat,customer_lng,distance_km
0,249e44947c66e3b0e46b3ba2f38d846d,22920e925fcbdd68ef776651f8c53cc3,delivered,2018-04-30 19:56:03,2018-05-01 02:55:24,2018-05-02 12:20:00,2018-05-09 20:14:48,2018-05-29,1.0,d1c427060a0f73f6b889a5c7c61f2ac4,...,0,28,0.165882,0,0.054369,-20.940578,-45.827237,-20.760206,-41.535163,446.440057
1,99e4c7eebc4e218f0ef4729921038c19,7b7d563426a10b8b26bfc679be70fbd8,delivered,2018-03-11 19:30:33,2018-03-11 19:47:54,2018-03-13 21:36:41,2018-04-08 13:37:48,2018-04-03,1.0,d48bacc1dcd9c86bf1ed4ed2a303336c,...,1,22,0.510690,0,0.171429,-23.049552,-47.837621,-19.655066,-43.233958,607.997754
2,be754126110440ba89a5456475333453,24c2d36dcc7f3ff6e41738c77dcf2def,delivered,2018-03-02 15:27:23,2018-03-06 03:55:55,2018-03-20 15:37:48,2018-03-29 19:21:58,2018-03-28,5.0,d6e74e35591c053e5cbab04d84c223b5,...,0,25,0.256449,0,0.400000,-26.923253,-48.695841,-22.509186,-43.219827,739.298901
3,f6c9fe3ff737f5568e352b5b2afcf112,b354952c4607431ab98d03af79f6d967,delivered,2018-08-23 19:28:09,2018-08-24 19:25:09,2018-08-27 14:46:00,2018-08-30 16:58:52,2018-09-18,1.0,67bd616e1ba0d3d3e8545f3113b0140d,...,0,25,1.258109,0,0.070039,-24.008923,-46.419125,-22.531717,-44.203700,279.622736
4,023669233121f0fb7899e5be2b22885f,22c15b46adce8afdd7d58e6582752263,delivered,2018-04-16 13:14:09,2018-04-17 04:54:27,2018-04-18 20:23:49,2018-04-19 16:37:46,2018-05-02,1.0,6b6b162b177d0f36987993aecbe1c65f,...,0,15,0.211383,1,0.125000,-23.588438,-46.511441,-23.514768,-46.502508,8.242176


In [29]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77108 entries, 0 to 77107
Data columns (total 42 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       77108 non-null  object        
 1   customer_id                    77108 non-null  object        
 2   order_status                   77108 non-null  object        
 3   order_purchase_timestamp       77108 non-null  datetime64[ns]
 4   order_approved_at              77096 non-null  datetime64[ns]
 5   order_delivered_carrier_date   77107 non-null  datetime64[ns]
 6   order_delivered_customer_date  77108 non-null  datetime64[ns]
 7   order_estimated_delivery_date  77108 non-null  datetime64[ns]
 8   order_item_id                  77108 non-null  float64       
 9   product_id                     77108 non-null  object        
 10  seller_id                      77108 non-null  object        
 11  shipping_limit_

In [30]:
train.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'is_late',
       'purchase_month', 'purchase_day', 'is_weekend',
       'estimated_delivery_days', 'freight_to_price_ratio', 'same_state',
       'seller_delay_rate', 'seller_lat', 'seller_lng', 'customer_lat',
       'customer_lng', 'distance_km'],
      dtype='object')

#validation


In [31]:
new_features = [
    'purchase_month',
    'purchase_day',
    'is_weekend',
    'estimated_delivery_days',
    'freight_to_price_ratio',
    'same_state'
]

train[new_features].head()

,purchase_month,purchase_day,is_weekend,estimated_delivery_days,freight_to_price_ratio,same_state
0,4,Monday,0,28,0.165882,0
1,3,Sunday,1,22,0.510690,0
2,3,Friday,0,25,0.256449,0
3,8,Thursday,0,25,1.258109,0
4,4,Monday,0,15,0.211383,1


In [32]:
train[new_features].isnull().sum()

purchase_month             0
purchase_day               0
is_weekend                 0
estimated_delivery_days    0
freight_to_price_ratio     0
same_state                 0
dtype: int64

In [33]:
train[new_features].dtypes


purchase_month               int32
purchase_day                object
is_weekend                   int64
estimated_delivery_days      int64
freight_to_price_ratio     float64
same_state                   int64
dtype: object

In [34]:
train[['purchase_month',
       'purchase_day',
       'is_weekend',
       'estimated_delivery_days',
       'freight_to_price_ratio',
       'same_state']].describe(include='all')

,purchase_month,purchase_day,is_weekend,estimated_delivery_days,freight_to_price_ratio,same_state
count,77108.000000,77108,77108.000000,77108.000000,77108.000000,77108.000000
unique,NaN,7,NaN,NaN,NaN,NaN
top,NaN,Tuesday,NaN,NaN,NaN,NaN
freq,NaN,12640,NaN,NaN,NaN,NaN
mean,6.026093,NaN,0.224763,23.420929,0.320523,0.362245
std,3.225693,NaN,0.417429,8.837262,0.337725,0.480652
min,1.000000,NaN,0.000000,2.000000,0.000000,0.000000
25%,3.000000,NaN,0.000000,18.000000,0.134832,0.000000
50%,6.000000,NaN,0.000000,23.000000,0.231701,0.000000
75%,8.000000,NaN,0.000000,28.000000,0.394874,1.000000


In [35]:
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

In [36]:
# creating seller_delay_rate
seller_delay_rate = (train.groupby('seller_id')['is_late'].mean().reset_index())
seller_delay_rate.rename(columns={'is_late':'seller_delay_rate'}, inplace=True)
seller_delay_rate.head()

,seller_id,seller_delay_rate
0,0015a82c2db000af6aaaf3ae2ecb0532,0.000000
1,001cca7ae9ae17fb1caed9dfb1094831,0.047337
2,002100f778ceb8431b7a1020ff7ab48f,0.200000
3,003554e2dce176b5555353e4f3555ac8,0.000000
4,004c9cd9d87a3c30c522c48c4fc07416,0.083333


In [37]:
# Merge Seller Delay Rate
train = train.merge(seller_delay_rate,on='seller_id',how='left')
train[['seller_id', 'seller_delay_rate']].head()

KeyError: "['seller_delay_rate'] not in index"

In [ ]:
train.shape

(77108, 37)

In [ ]:
train.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'is_late',
 'purchase_month',
 'purchase_day',
 'is_weekend',
 'estimated_delivery_days',
 'freight_to_price_ratio',
 'same_state',
 'seller_delay_rate']

In [ ]:
geo = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

In [ ]:
print(geo.shape)

(1000163, 5)


In [ ]:
geo.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [ ]:
geo.columns.tolist()

['geolocation_zip_code_prefix',
 'geolocation_lat',
 'geolocation_lng',
 'geolocation_city',
 'geolocation_state']

In [ ]:
# Create unique geolocation dataset
geo_unique = (geo.groupby('geolocation_zip_code_prefix', as_index=False).agg({'geolocation_lat':'mean','geolocation_lng':'mean'}))
print('original geolocation shape : ',geo.shape)
print('unique geolocation shape : ',geo_unique.shape)
geo_unique.head()


original geolocation shape :  (1000163, 5)
unique geolocation shape :  (19015, 3)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,1001,-23.550190,-46.634024
1,1002,-23.548146,-46.634979
2,1003,-23.548994,-46.635731
3,1004,-23.549799,-46.634757
4,1005,-23.549456,-46.636733


In [ ]:
# Merge Seller Coordinates
seller_geo = geo_unique.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "geolocation_lat": "seller_lat",
    "geolocation_lng": "seller_lng"
})
train = train.merge(seller_geo, on='seller_zip_code_prefix', how='left')
print(train.shape)
train[["seller_zip_code_prefix", "seller_lat", "seller_lng"]].head()

(77108, 39)


,seller_zip_code_prefix,seller_lat,seller_lng
0,37175.0,-20.940578,-45.827237
1,18500.0,-23.049552,-47.837621
2,88308.0,-26.923253,-48.695841
3,11701.0,-24.008923,-46.419125
4,3916.0,-23.588438,-46.511441


In [ ]:
# Merge Customer Coordinates
customer_geo = (geo_unique.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng"
    }))
train = train.merge(customer_geo, on="customer_zip_code_prefix", how='left')
print(train.shape)
train[["customer_zip_code_prefix","customer_lat","customer_lng"]].head()

(77108, 41)


,customer_zip_code_prefix,customer_lat,customer_lng
0,29500,-20.760206,-41.535163
1,35901,-19.655066,-43.233958
2,25665,-22.509186,-43.219827
3,27351,-22.531717,-44.203700
4,3673,-23.514768,-46.502508


In [ ]:
# calculating distance feature
from math import sin,cos,atan2,sqrt,radians

def haversine(lat1,lon1,lat2,lon2):
    R = 6371 #Earths radius in kilometers
    lat1,lon1,lat2,lon2 = map(radians,[lat1,lon1,lat2,lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2)
    c = 2 * atan2(sqrt(a),sqrt(1-a))
    return R * c
train['distance_km'] = train.apply(
    lambda row : haversine(
        row['seller_lat'],
        row['seller_lng'],
        row['customer_lat'],
        row['customer_lng'],
    ),axis=1
)
train[['seller_lat','seller_lng','customer_lat','customer_lng','distance_km']].head()

,seller_lat,seller_lng,customer_lat,customer_lng,distance_km
0,-20.940578,-45.827237,-20.760206,-41.535163,446.440057
1,-23.049552,-47.837621,-19.655066,-43.233958,607.997754
2,-26.923253,-48.695841,-22.509186,-43.219827,739.298901
3,-24.008923,-46.419125,-22.531717,-44.203700,279.622736
4,-23.588438,-46.511441,-23.514768,-46.502508,8.242176


In [ ]:
# checking missing values for new features
NewFeatures = ["seller_delay_rate","seller_lat","seller_lng","customer_lat","customer_lng","distance_km"]
print(train[NewFeatures].isnull().sum())

seller_delay_rate      0
seller_lat           183
seller_lng           183
customer_lat         193
customer_lng         193
distance_km          375
dtype: int64


In [ ]:
# Rows where seller coordinates are missing
train[train["seller_lat"].isnull()][["seller_zip_code_prefix", "seller_city", "seller_state"]].head(10)

,seller_zip_code_prefix,seller_city,seller_state
1083,37708.0,pocos de caldas,MG
1114,2285.0,sao paulo,SP
1592,71551.0,brasilia,DF
2743,71551.0,brasilia,DF
2800,2285.0,sao paulo,SP
2885,82040.0,curitiba,PR
2974,2285.0,sao paulo,SP
3513,37708.0,pocos de caldas,MG
3590,2285.0,sao paulo,SP
3820,2285.0,sao paulo,SP


In [ ]:
# Rows where customer coordinates are missing
train[train["customer_lat"].isnull()][["customer_zip_code_prefix", "customer_city", "customer_state"]].head(10)

,customer_zip_code_prefix,customer_city,customer_state
14,72017,brasilia,DF
69,7729,caieiras,SP
770,73093,brasilia,DF
775,71591,brasilia,DF
1456,70686,brasilia,DF
1488,7430,aruja,SP
2291,73272,brasilia,DF
2983,73369,brasilia,DF
3143,28530,sao sebastiao do paraiba,RJ
3220,71539,brasilia,DF


### Feature Validation

After creating seller coordinates, customer coordinates, and distance_km:

Missing Values:

- seller_lat : 183
- seller_lng : 183
- customer_lat : 193
- customer_lng : 193
- distance_km : 375

Reason:

Some seller/customer ZIP code prefixes do not have matching records in the geolocation dataset. This results in missing coordinates and therefore missing distance values.

In [ ]:
# Merge Seller Delay Rate into Test

test = test.merge(seller_delay_rate,on="seller_id",how="left")
print(test.shape)
test[["seller_id","seller_delay_rate"]].head()

(17014, 36)


,seller_id,seller_delay_rate
0,1025f0e2d44d7041d6cf58b6550e0bfa,0.089447
1,8b321bb669392f5163d04c59e235e066,0.100282
2,1127b7f2594683f2510f1c2c834a486b,0.078947
3,512d298ac2a96d1931b6bd30aa21f61d,0.000000
4,04308b1ee57b6625f47df1d56f00eedf,0.111111


In [ ]:
# Merge Seller Coordinates into Test
seller_geo = geo_unique.rename(columns={"geolocation_zip_code_prefix": "seller_zip_code_prefix",
                                        "geolocation_lat": "seller_lat","geolocation_lng": "seller_lng"})
test = test.merge(seller_geo,on="seller_zip_code_prefix",how="left")
print(test.shape)
test[["seller_zip_code_prefix","seller_lat","seller_lng"]].head()

(17014, 38)


,seller_zip_code_prefix,seller_lat,seller_lng
0,3204.0,-23.595499,-46.559727
1,1212.0,-23.538269,-46.639423
2,13087.0,-22.857990,-47.053051
3,20060.0,-22.907426,-43.221113
4,88215.0,-27.156573,-48.503204


In [ ]:
# Merge Customer Coordinates into Test

customer_geo = geo_unique.rename(columns={"geolocation_zip_code_prefix": "customer_zip_code_prefix",
                                          "geolocation_lat": "customer_lat","geolocation_lng": "customer_lng"})
test = test.merge(customer_geo,on="customer_zip_code_prefix",how="left")
print(test.shape)
test[["customer_zip_code_prefix","customer_lat","customer_lng"]].head()

(17014, 40)


,customer_zip_code_prefix,customer_lat,customer_lng
0,12240,-23.224298,-45.913201
1,64160,-3.461298,-42.369978
2,24743,-22.845998,-42.997068
3,35613,-19.532003,-45.787813
4,69301,2.822775,-60.670698


In [ ]:
# Calculate Distance Feature for Test
test["distance_km"] = test.apply(
    lambda row: haversine(
        row["seller_lat"],
        row["seller_lng"],
        row["customer_lat"],
        row["customer_lng"],
    ),axis=1,
)
print(test.shape)
test[["seller_lat","seller_lng","customer_lat","customer_lng","distance_km",]].head()

(17014, 41)


,seller_lat,seller_lng,customer_lat,customer_lng,distance_km
0,-23.595499,-46.559727,-23.224298,-45.913201,77.820679
1,-23.538269,-46.639423,-3.461298,-42.369978,2279.120568
2,-22.857990,-47.053051,-22.845998,-42.997068,415.594947
3,-22.907426,-43.221113,-19.532003,-45.787813,460.027927
4,-27.156573,-48.503204,2.822775,-60.670698,3579.894251


In [ ]:
# check missing values in test dataset
newfeatures = [
    "seller_delay_rate",
    "seller_lat",
    "seller_lng",
    "customer_lat",
    "customer_lng",
    "distance_km"
]

print(test[newfeatures].isnull().sum())

seller_delay_rate    273
seller_lat           153
seller_lng           153
customer_lat          37
customer_lng          37
distance_km          190
dtype: int64


In [ ]:
# Rows where seller coordinates are missing
test[test["seller_lat"].isnull()][["seller_zip_code_prefix", "seller_city", "seller_state"]].head(10)

,seller_zip_code_prefix,seller_city,seller_state
204,NaN,NaN,NaN
264,2285.0,sao paulo,SP
358,NaN,NaN,NaN
372,2285.0,sao paulo,SP
672,NaN,NaN,NaN
797,NaN,NaN,NaN
947,NaN,NaN,NaN
1029,2285.0,sao paulo,SP
1033,NaN,NaN,NaN
1174,NaN,NaN,NaN


In [ ]:
# Rows where customer coordinates are missing
test[test["customer_lat"].isnull()][["customer_zip_code_prefix", "customer_city", "customer_state"]].head(10)

,customer_zip_code_prefix,customer_city,customer_state
421,70686,brasilia,DF
1575,71551,brasilia,DF
1634,70686,brasilia,DF
2745,72005,brasilia,DF
2773,65137,maioba,MA
3493,73401,brasilia,DF
3539,70324,brasilia,DF
3589,71810,brasilia,DF
3618,72341,brasilia,DF
4302,72280,brasilia,DF


In [ ]:
# Calculate median values from training data

seller_lat_median = train["seller_lat"].median()
seller_lng_median = train["seller_lng"].median()
customer_lat_median = train["customer_lat"].median()
customer_lng_median = train["customer_lng"].median()
distance_median = train["distance_km"].median()

print("Seller Latitude Median :", seller_lat_median)
print("Seller Longitude Median:", seller_lng_median)
print("Customer Latitude Median :", customer_lat_median)
print("Customer Longitude Median:", customer_lng_median)
print("Distance Median (km):", distance_median)

Seller Latitude Median : -23.425555705618997
Seller Longitude Median: -46.744091586764384
Customer Latitude Median : -22.928999413639033
Customer Longitude Median: -46.63498468822016
Distance Median (km): 431.68258878849787


In [ ]:
# Fill Missing Values in Training Set

train["seller_lat"] = train["seller_lat"].fillna(seller_lat_median)
train["seller_lng"] = train["seller_lng"].fillna(seller_lng_median)

train["customer_lat"] = train["customer_lat"].fillna(customer_lat_median)
train["customer_lng"] = train["customer_lng"].fillna(customer_lng_median)

train["distance_km"] = train["distance_km"].fillna(distance_median)

In [ ]:
NewFeatures = [
    "seller_delay_rate",
    "seller_lat",
    "seller_lng",
    "customer_lat",
    "customer_lng",
    "distance_km"
]

print(train[NewFeatures].isnull().sum())

seller_delay_rate    0
seller_lat           0
seller_lng           0
customer_lat         0
customer_lng         0
distance_km          0
dtype: int64


In [ ]:
# Fill missing values in test dataset

test["seller_delay_rate"] = test["seller_delay_rate"].fillna(0)

test["seller_lat"] = test["seller_lat"].fillna(seller_lat_median)
test["seller_lng"] = test["seller_lng"].fillna(seller_lng_median)

test["customer_lat"] = test["customer_lat"].fillna(customer_lat_median)
test["customer_lng"] = test["customer_lng"].fillna(customer_lng_median)

test["distance_km"] = test["distance_km"].fillna(distance_median)

In [ ]:
print(test[newfeatures].isnull().sum())

seller_delay_rate    0
seller_lat           0
seller_lng           0
customer_lat         0
customer_lng         0
distance_km          0
dtype: int64


In [ ]:
new_features = [
    "seller_delay_rate",
    "seller_lat",
    "seller_lng",
    "customer_lat",
    "customer_lng",
    "distance_km"
]

train[new_features].head()
test[new_features].head()

,seller_delay_rate,seller_lat,seller_lng,customer_lat,customer_lng,distance_km
0,0.089447,-23.595499,-46.559727,-23.224298,-45.913201,77.820679
1,0.100282,-23.538269,-46.639423,-3.461298,-42.369978,2279.120568
2,0.078947,-22.857990,-47.053051,-22.845998,-42.997068,415.594947
3,0.000000,-22.907426,-43.221113,-19.532003,-45.787813,460.027927
4,0.111111,-27.156573,-48.503204,2.822775,-60.670698,3579.894251


In [ ]:
train[new_features].dtypes

seller_delay_rate    float64
seller_lat           float64
seller_lng           float64
customer_lat         float64
customer_lng         float64
distance_km          float64
dtype: object

In [ ]:
test[new_features].dtypes

seller_delay_rate    float64
seller_lat           float64
seller_lng           float64
customer_lat         float64
customer_lng         float64
distance_km          float64
dtype: object

In [ ]:
train[new_features].describe()

,seller_delay_rate,seller_lat,seller_lng,customer_lat,customer_lng,distance_km
count,77108.000000,77108.000000,77108.000000,77108.000000,77108.000000,77108.000000
mean,0.079758,-22.800519,-47.239325,-21.252267,-46.220972,593.596923
std,0.068870,2.691641,2.334720,5.536256,4.019672,586.371530
min,0.000000,-32.079231,-63.893565,-33.689948,-72.668881,0.000000
25%,0.043478,-23.607473,-48.829744,-23.589483,-48.118347,185.708734
50%,0.070588,-23.425556,-46.744092,-22.928999,-46.634985,431.682589
75%,0.101562,-21.757321,-46.522287,-20.188080,-43.700266,785.340686
max,1.000000,-2.501242,-34.855616,42.184003,-8.723762,8677.911622


In [ ]:
test[new_features].describe()

,seller_delay_rate,seller_lat,seller_lng,customer_lat,customer_lng,distance_km
count,17014.000000,17014.000000,17014.000000,17014.000000,17014.000000,17014.000000
mean,0.078317,-22.789412,-47.237525,-21.172780,-46.185295,607.868223
std,0.069675,2.695073,2.355236,5.683148,4.087928,596.046317
min,0.000000,-32.079231,-61.958415,-33.689948,-72.668881,0.000000
25%,0.041667,-23.600627,-48.829744,-23.592764,-48.122730,204.063879
50%,0.068796,-23.425556,-46.750602,-22.927919,-46.634985,433.566602
75%,0.101124,-21.757321,-46.523183,-20.073479,-43.595250,801.532596
max,1.000000,-2.501242,-34.855616,2.855558,-34.823063,3927.406027


In [ ]:
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

In [ ]:
print(train.shape)
print(test.shape)

(77108, 42)
(17014, 41)


In [ ]:
print(train.columns.tolist())
print(test.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'is_late', 'purchase_month', 'purchase_day', 'is_weekend', 'estimated_delivery_days', 'freight_to_price_ratio', 'same_state', 'seller_delay_rate', 'seller_lat', 'seller_lng', 'customer_lat', 'customer_lng', 'distance_km']
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estim

In [44]:
df = pd.read_csv("../data/processed/train.csv")
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,is_weekend,estimated_delivery_days,freight_to_price_ratio,same_state,seller_delay_rate,seller_lat,seller_lng,customer_lat,customer_lng,distance_km
0,249e44947c66e3b0e46b3ba2f38d846d,22920e925fcbdd68ef776651f8c53cc3,delivered,2018-04-30 19:56:03,2018-05-01 02:55:24,2018-05-02 12:20:00,2018-05-09 20:14:48,2018-05-29,1.0,d1c427060a0f73f6b889a5c7c61f2ac4,...,0,28,0.165882,0,0.054369,-20.940578,-45.827237,-20.760206,-41.535163,446.440057
1,99e4c7eebc4e218f0ef4729921038c19,7b7d563426a10b8b26bfc679be70fbd8,delivered,2018-03-11 19:30:33,2018-03-11 19:47:54,2018-03-13 21:36:41,2018-04-08 13:37:48,2018-04-03,1.0,d48bacc1dcd9c86bf1ed4ed2a303336c,...,1,22,0.510690,0,0.171429,-23.049552,-47.837621,-19.655066,-43.233958,607.997754
2,be754126110440ba89a5456475333453,24c2d36dcc7f3ff6e41738c77dcf2def,delivered,2018-03-02 15:27:23,2018-03-06 03:55:55,2018-03-20 15:37:48,2018-03-29 19:21:58,2018-03-28,5.0,d6e74e35591c053e5cbab04d84c223b5,...,0,25,0.256449,0,0.400000,-26.923253,-48.695841,-22.509186,-43.219827,739.298901
3,f6c9fe3ff737f5568e352b5b2afcf112,b354952c4607431ab98d03af79f6d967,delivered,2018-08-23 19:28:09,2018-08-24 19:25:09,2018-08-27 14:46:00,2018-08-30 16:58:52,2018-09-18,1.0,67bd616e1ba0d3d3e8545f3113b0140d,...,0,25,1.258109,0,0.070039,-24.008923,-46.419125,-22.531717,-44.203700,279.622736
4,023669233121f0fb7899e5be2b22885f,22c15b46adce8afdd7d58e6582752263,delivered,2018-04-16 13:14:09,2018-04-17 04:54:27,2018-04-18 20:23:49,2018-04-19 16:37:46,2018-05-02,1.0,6b6b162b177d0f36987993aecbe1c65f,...,0,15,0.211383,1,0.125000,-23.588438,-46.511441,-23.514768,-46.502508,8.242176
